# 12B — Rescue MedGemma formatting failures

**DISABLED in generation v3.** Keep old rescue artifacts for historical audits. Use Notebook 12 (CXR only) with a new E4_GENERATION_RUN for a complete, isolated comparison. Do not combine old successes with revised retries. run_format_rescue deliberately stops before loading a model or writing results.

In [ ]:
from pathlib import Path
import os, sys
CODE_ROOT = Path(os.environ.get('EXPERIMENT4_CODE_ROOT', str(Path.cwd())))
if not (CODE_ROOT / 'e4_common.py').is_file() and 'EXPERIMENT4_CODE_ROOT' not in os.environ:
    CODE_ROOT = Path.cwd()
if not CODE_ROOT.is_absolute() or not (CODE_ROOT / 'e4_common.py').is_file():
    raise FileNotFoundError('Set EXPERIMENT4_CODE_ROOT to the uploaded Experiment 4 source directory')
if str(CODE_ROOT) not in sys.path: sys.path.insert(0, str(CODE_ROOT))
from e4_common import ARTIFACT_ROOT, EXPERIMENT4_ROOT, SYMILE_ROOT
print({'code_root': str(CODE_ROOT), 'artifact_root': str(ARTIFACT_ROOT),
       'experiment4_root': str(EXPERIMENT4_ROOT), 'symile_root': str(SYMILE_ROOT)})

In [ ]:
from huggingface_hub import HfApi, login
token_path = Path(os.environ['HF_TOKEN_FILE']).expanduser()  # Explicit private file; historical workflow only.
hf_token = token_path.read_text().strip()
if not hf_token or any(char.isspace() for char in hf_token):
    raise ValueError(f'Token file must contain exactly one token: {token_path}')
login(token=hf_token, add_to_git_credential=False)
model_info = HfApi().model_info('google/medgemma-1.5-4b-it', token=hf_token)
print({'model_repo': 'google/medgemma-1.5-4b-it', 'revision': model_info.sha, 'token_source': str(token_path)})

In [ ]:
from e4_format_rescue import audit_rescue
audit = audit_rescue('medgemma15_4b')
audit

In [ ]:
from e4_format_rescue import run_format_rescue
MAX_ROWS = int(os.environ.get('E4_RESCUE_MAX_ROWS', '50'))
REPAIR_ATTEMPTS = int(os.environ.get('E4_RESCUE_REPAIR_ATTEMPTS', '2'))
RETRY_TERMINAL = os.environ.get('E4_RESCUE_RETRY_TERMINAL', '0').lower() in {'1', 'true', 'yes'}
result = run_format_rescue('medgemma15_4b', max_rows=MAX_ROWS, repair_attempts=REPAIR_ATTEMPTS,
                           retry_terminal=RETRY_TERMINAL)
result